# XClinVision: Explainability Demo (Grad-CAM++)
**Day 5: Explainability Integration**

This notebook demonstrates:
- Grad-CAM++ heatmap generation
- Region importance scoring
- Attention visualization for clinical interpretability

## 1. Setup and Imports

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path().absolute().parent / "src"))

import numpy as np
import matplotlib.pyplot as plt
import cv2
from PIL import Image
import torch

from xclinvision.xai import ExplainabilityEngine
from xclinvision.architecture import create_model

plt.rcParams["figure.figsize"] = (14, 10)

## 2. Load Sample Image

In [ ]:
# Find a sample image
DATA_DIR = Path("../data/raw")

sample_img = None
sample_class = None

for dataset_dir in DATA_DIR.iterdir():
    if dataset_dir.is_dir():
        for split in ["train", "val", "test"]:
            split_dir = dataset_dir / split
            if split_dir.exists():
                for class_dir in split_dir.iterdir():
                    if class_dir.is_dir():
                        img_files = list(class_dir.glob("*.jpeg"))
                        if img_files:
                            sample_img = img_files[0]
                            sample_class = class_dir.name
                            break
                if sample_img:
                    break
        if sample_img:
            break

if sample_img:
    print(f"Sample: {sample_img}")
    print(f"Class: {sample_class}")
    
    # Load and preprocess
    img = Image.open(sample_img).convert("RGB")
    img_np = np.array(img)
    
    plt.figure(figsize=(8, 8))
    plt.imshow(img_np)
    plt.title(f"Original Image - {sample_class}")
    plt.axis("off")
    plt.show()
else:
    print("No sample images found. Please download datasets first.")

## 3. Grad-CAM++ Demo with Simulated Heatmap
*(In practice, use actual model activations)*

In [ ]:
def simulate_gradcam_heatmap(image_shape, focus_regions=None):
    """Simulate a Grad-CAM++ heatmap with regional focus."""
    
    h, w = image_shape[:2]
    heatmap = np.zeros((h, w), dtype=np.float32)
    
    if focus_regions is None:
        # Default: simulate focus on lungs (center regions)
        focus_regions = [
            (h//4, h//2, w//4, 3*w//4),   # Upper lungs
            (h//2, 3*h//4, w//3, 2*w//3),  # Lower lungs
        ]
    
    # Add Gaussian blobs at focus regions
    for y1, y2, x1, x2 in focus_regions:
        center_y, center_x = (y1 + y2) // 2, (x1 + x2) // 2
        sigma_y, sigma_x = (y2 - y1) // 4, (x2 - x1) // 4
        
        y, x = np.ogrid[:h, :w]
        gaussian = np.exp(-((y - center_y)**2 / (2 * sigma_y**2) + 
                           (x - center_x)**2 / (2 * sigma_x**2)))
        heatmap += gaussian
    
    # Normalize
    heatmap = (heatmap - heatmap.min()) / (heatmap.max() - heatmap.min() + 1e-8)
    
    return heatmap

if sample_img:
    # Generate simulated heatmap
    heatmap = simulate_gradcam_heatmap(img_np.shape)
    
    # Apply colormap
    heatmap_colored = plt.cm.jet(heatmap)[:, :, :3]
    heatmap_colored = (heatmap_colored * 255).astype(np.uint8)
    
    # Overlay on original
    alpha = 0.5
    overlay = (img_np * (1 - alpha) + heatmap_colored * alpha).astype(np.uint8)
    
    # Visualize
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    axes[0].imshow(img_np)
    axes[0].set_title("Original Image")
    axes[0].axis("off")
    
    axes[1].imshow(heatmap, cmap="jet")
    axes[1].set_title("Grad-CAM++ Heatmap")
    axes[1].axis("off")
    
    axes[2].imshow(overlay)
    axes[2].set_title(f"Overlay (alpha={alpha})")
    axes[2].axis("off")
    
    plt.tight_layout()
    plt.show()

## 4. Region Importance Scoring

In [ ]:
def compute_region_scores(heatmap, n_regions=5):
    """Compute importance scores for anatomical regions."""
    
    h, w = heatmap.shape
    
    # Define regions
    regions = {
        "Left Upper": (0, h//2, 0, w//2),
        "Right Upper": (0, h//2, w//2, w),
        "Left Lower": (h//2, h, 0, w//2),
        "Right Lower": (h//2, h, w//2, w),
        "Center (Mediastinum)": (h//3, 2*h//3, w//3, 2*w//3),
    }
    
    scores = {}
    for name, (y1, y2, x1, x2) in regions.items():
        region_heatmap = heatmap[y1:y2, x1:x2]
        scores[name] = {
            "mean": region_heatmap.mean(),
            "max": region_heatmap.max(),
            "percentile_95": np.percentile(region_heatmap, 95),
        }
    
    return scores

if sample_img:
    region_scores = compute_region_scores(heatmap)
    
    print("\nRegion Importance Scores:")
    print("=" * 50)
    
    # Sort by mean score
    sorted_regions = sorted(region_scores.items(), key=lambda x: x[1]["mean"], reverse=True)
    
    for i, (region, scores) in enumerate(sorted_regions, 1):
        print(f"\n{i}. {region}:")
        print(f"   Mean: {scores['mean']:.3f}")
        print(f"   Max: {scores['max']:.3f}")
        print(f"   95th percentile: {scores['percentile_95']:.3f}")

## 5. Key Findings Extraction

In [ ]:
def extract_key_findings(region_scores, threshold=0.5):
    """Extract key findings from region scores."""
    
    findings = []
    
    # Sort by importance
    sorted_regions = sorted(region_scores.items(), key=lambda x: x[1]["mean"], reverse=True)
    
    for region, scores in sorted_regions:
        if scores["mean"] > threshold:
            findings.append(f"High activation in {region} (score: {scores['mean']:.2f})")
    
    return findings

if sample_img:
    findings = extract_key_findings(region_scores, threshold=0.3)
    
    print("\n🔍 Key Findings:")
    print("=" * 50)
    for finding in findings:
        print(f"  • {finding}")
    
    if not findings:
        print("  • No significant regions detected above threshold")

## 6. Multi-Class Explainability

In [ ]:
if sample_img:
    # Simulate heatmaps for each class
    classes = ["Normal", "Pneumonia", "Cardiomegaly"]
    
    # Different focus regions for each class
    class_regions = {
        "Normal": [
            (96, 192, 96, 288),   # Center clear
        ],
        "Pneumonia": [
            (96, 192, 48, 144),   # Right upper (consolidation)
            (192, 288, 192, 336), # Left lower
        ],
        "Cardiomegaly": [
            (48, 144, 48, 144),   # Right apex
            (48, 144, 240, 336),  # Left apex
        ],
    }
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    
    for i, class_name in enumerate(classes):
        # Generate class-specific heatmap
        regions = class_regions.get(class_name, [])
        hm = simulate_gradcam_heatmap(img_np.shape, focus_regions=regions)
        
        # Original
        axes[0, i].imshow(img_np)
        axes[0, i].set_title(f"{class_name} - Original")
        axes[0, i].axis("off")
        
        # Heatmap
        axes[1, i].imshow(hm, cmap="jet")
        axes[1, i].set_title(f"{class_name} - Explanation")
        axes[1, i].axis("off")
    
    plt.suptitle("Multi-Class Explainability Comparison", fontsize=16)
    plt.tight_layout()
    plt.show()